<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from google.colab import drive

# ============================================================
# 1. 데이터 로드
# ============================================================
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx'
df_raw = pd.read_excel(file_path)


# ============================================================
# 2. 비율 변수 변환 함수
# ============================================================
def convert_to_ratio(series):
    """
    0~100 단위의 비율이면 0~1로 변환하고,
    이미 0~1 단위이면 그대로 유지
    """
    series = pd.to_numeric(series, errors="coerce")

    valid = series.dropna()

    if len(valid) == 0:
        return series

    if valid.abs().max() > 2.0:
        return series / 100.0

    return series


# ============================================================
# 3. 전처리 함수
# ============================================================
def prepare_seoul_df_final(df_input):
    df = df_input.copy()

    # --------------------------------------------------------
    # 연속형 변수 숫자 변환
    # --------------------------------------------------------
    numeric_cols = [
        "Log_Price_per_m2",
        "Size_m2",
        "Floor",
        "num_of_people",
        "Parking_per_Household",
        "Construction_Year",
        "max_floor",
        "Log_Dist_Subway",
        "Log_Dist_Green",
        "Log_Dist_Water",
        "Dist_CBD",
        "Bus_Stop",
        "High_School_Count",
        "Population",
        "Sex_ratio",
        "Pop. Density",
        "Median age Population",
        "Old Population",
        "Young Population",
        "Year_Sold",
        "Month_Sold"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # --------------------------------------------------------
    # 변수명 정리
    # --------------------------------------------------------
    if "Pop. Density" in df.columns:
        df["Pop_Density"] = df["Pop. Density"]

    if "Old Population" in df.columns:
        df["Old_pop_ratio"] = convert_to_ratio(df["Old Population"])

    if "Young Population" in df.columns:
        df["Young_pop_ratio"] = convert_to_ratio(df["Young Population"])

    # --------------------------------------------------------
    # 구청까지의 거리: m → km
    # --------------------------------------------------------
    if "Dist_CBD" in df.columns:
        df["Dist_CBD_km"] = df["Dist_CBD"] / 1000.0

    # --------------------------------------------------------
    # 난방방식 더미변수
    # 도시가스 = 1, 그 외 = 0
    # --------------------------------------------------------
    if "heating" in df.columns:

        if pd.api.types.is_numeric_dtype(df["heating"]):
            df["Heating_Dummy"] = pd.to_numeric(
                df["heating"],
                errors="coerce"
            )

        else:
            heating_missing = df["heating"].isna()

            df["Heating_Dummy"] = (
                df["heating"]
                .astype(str)
                .str.contains("도시가스", na=False)
                .astype(float)
            )

            df.loc[heating_missing, "Heating_Dummy"] = np.nan

    # --------------------------------------------------------
    # 계절 더미변수
    # 여름(6·7·8월)을 기준범주로 설정
    # --------------------------------------------------------
    if "Month_Sold" in df.columns:
        month = pd.to_numeric(df["Month_Sold"], errors="coerce")

        df["Spring"] = month.isin([3, 4, 5]).astype(int)
        df["Fall"] = month.isin([9, 10, 11]).astype(int)
        df["Winter"] = month.isin([12, 1, 2]).astype(int)

    else:
        df["Spring"] = 0
        df["Fall"] = 0
        df["Winter"] = 0

    return df


# ============================================================
# 4. 랜덤포레스트 실행 함수
# ============================================================
def run_random_forest_prediction(
    df_sub,
    feature_cols,
    target_col="Log_Price_per_m2",
    random_state=42
):
    """
    Random Forest 회귀모형

    - 훈련 데이터셋 70%
    - 테스트 데이터셋 30%
    - 훈련 및 테스트 성능 산출
    - 테스트 예측값 저장
    """

    df_model = df_sub.copy()

    # --------------------------------------------------------
    # 필요한 변수 존재 여부 확인
    # --------------------------------------------------------
    missing_cols = [
        col for col in feature_cols + [target_col]
        if col not in df_model.columns
    ]

    if missing_cols:
        raise ValueError(
            f"다음 변수가 데이터에 없습니다: {missing_cols}"
        )

    # --------------------------------------------------------
    # 숫자형 변환
    # --------------------------------------------------------
    for col in feature_cols + [target_col]:
        df_model[col] = pd.to_numeric(
            df_model[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # 결측치 제거
    # --------------------------------------------------------
    df_model = (
        df_model
        .dropna(subset=feature_cols + [target_col])
        .reset_index(drop=True)
    )

    X = df_model[feature_cols]
    y = df_model[target_col]

    # --------------------------------------------------------
    # 훈련 데이터셋 70%, 테스트 데이터셋 30%
    # --------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_state
    )

    # --------------------------------------------------------
    # 랜덤포레스트 모형
    # --------------------------------------------------------
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=random_state,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    # --------------------------------------------------------
    # 훈련 데이터셋 성능
    # --------------------------------------------------------
    y_pred_train = rf.predict(X_train)

    r2_train = r2_score(y_train, y_pred_train)
    rmse_train = np.sqrt(
        mean_squared_error(y_train, y_pred_train)
    )

    # --------------------------------------------------------
    # 테스트 데이터셋 성능
    # --------------------------------------------------------
    y_pred_test = rf.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(
        mean_squared_error(y_test, y_pred_test)
    )

    # --------------------------------------------------------
    # 테스트 예측 결과
    # --------------------------------------------------------
    pred_df = pd.DataFrame({
        "Actual": y_test.to_numpy(),
        "Predicted": y_pred_test
    }).reset_index(drop=True)

    # --------------------------------------------------------
    # 변수 중요도
    # --------------------------------------------------------
    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": rf.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "model": rf,
        "r2_train": r2_train,
        "rmse_train": rmse_train,
        "r2_test": r2_test,
        "rmse_test": rmse_test,
        "pred_df": pred_df,
        "importance_df": importance_df,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "used_features": feature_cols
    }


# ============================================================
# 5. 데이터 준비
# ============================================================
df_all = prepare_seoul_df_final(df_raw)


# ============================================================
# 6. 최종 설명변수
# 위도와 경도는 제외
# ============================================================
features = [
    # 개별 주택 및 단지 특성
    "Size_m2",
    "Floor",
    "num_of_people",
    "Parking_per_Household",
    "Construction_Year",
    "max_floor",
    "Heating_Dummy",

    # 지역 접근성 및 주변 시설
    "Log_Dist_Subway",
    "Log_Dist_Green",
    "Log_Dist_Water",
    "Dist_CBD_km",
    "Bus_Stop",
    "High_School_Count",

    # 지역 인구 특성
    "Population",
    "Sex_ratio",
    "Pop_Density",
    "Median age Population",
    "Old_pop_ratio",
    "Young_pop_ratio",

    # 계절 통제변수
    "Spring",
    "Fall",
    "Winter"
]

target = "Log_Price_per_m2"

print("최종 설명변수 수:", len(features))
print(features)


# ============================================================
# 7. 연도별 랜덤포레스트
# ============================================================
rf_results_by_year = []
rf_prediction_dict = {}
rf_importance_dict = {}

for year in [2022, 2023]:

    df_year = df_all[
        pd.to_numeric(
            df_all["Year_Sold"],
            errors="coerce"
        ) == year
    ].copy()

    if len(df_year) < 30:
        print(f"{year}: 표본 수 부족으로 분석에서 제외")
        continue

    rf_out = run_random_forest_prediction(
        df_sub=df_year,
        feature_cols=features,
        target_col=target,
        random_state=42
    )

    rf_results_by_year.append({
        "Year": year,
        "Train_R2": rf_out["r2_train"],
        "Train_RMSE": rf_out["rmse_train"],
        "Test_R2": rf_out["r2_test"],
        "Test_RMSE": rf_out["rmse_test"],
        "Train_N": rf_out["n_train"],
        "Test_N": rf_out["n_test"]
    })

    rf_prediction_dict[year] = rf_out["pred_df"]
    rf_importance_dict[year] = rf_out["importance_df"]

    print(f"\n[{year}년 Random Forest 결과]")

    print("훈련 데이터셋")
    print(f"R²   : {rf_out['r2_train']:.4f}")
    print(f"RMSE : {rf_out['rmse_train']:.4f}")

    print("테스트 데이터셋")
    print(f"R²   : {rf_out['r2_test']:.4f}")
    print(f"RMSE : {rf_out['rmse_test']:.4f}")

    print(f"Train N : {rf_out['n_train']:,}")
    print(f"Test N  : {rf_out['n_test']:,}")

    print("\n변수 중요도 상위 10개")
    display(rf_out["importance_df"].head(10))


# ============================================================
# 8. 전체 기간 랜덤포레스트
# ============================================================
df_full = df_all.copy()

rf_full_out = run_random_forest_prediction(
    df_sub=df_full,
    feature_cols=features,
    target_col=target,
    random_state=42
)

print("\n[전체 기간 Random Forest 결과]")

print("훈련 데이터셋")
print(f"R²   : {rf_full_out['r2_train']:.4f}")
print(f"RMSE : {rf_full_out['rmse_train']:.4f}")

print("테스트 데이터셋")
print(f"R²   : {rf_full_out['r2_test']:.4f}")
print(f"RMSE : {rf_full_out['rmse_test']:.4f}")

print(f"Train N : {rf_full_out['n_train']:,}")
print(f"Test N  : {rf_full_out['n_test']:,}")

print("\n전체 기간 변수 중요도 상위 10개")
display(rf_full_out["importance_df"].head(10))


# ============================================================
# 9. 성능 비교표
# ============================================================
rf_results_table = pd.DataFrame(rf_results_by_year)

full_row = pd.DataFrame([{
    "Year": "Full Sample",
    "Train_R2": rf_full_out["r2_train"],
    "Train_RMSE": rf_full_out["rmse_train"],
    "Test_R2": rf_full_out["r2_test"],
    "Test_RMSE": rf_full_out["rmse_test"],
    "Train_N": rf_full_out["n_train"],
    "Test_N": rf_full_out["n_test"]
}])

rf_results_table = pd.concat(
    [rf_results_table, full_row],
    ignore_index=True
)

print("\n[Random Forest 성능 비교표]")
display(rf_results_table)


# ============================================================
# 10. 예측값 일부 확인
# ============================================================
for year, pred_df in rf_prediction_dict.items():

    print(f"\n[{year}년 예측값 일부]")
    display(pred_df.head())

print("\n[전체 기간 예측값 일부]")
display(rf_full_out["pred_df"].head())


# ============================================================
# 11. 엑셀 저장
# ============================================================
output_path = (
    "/content/drive/MyDrive/"
    "Seoul_Apartment_RF_Results_All_Features.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    # 성능 결과
    rf_results_table.to_excel(
        writer,
        sheet_name="RF_Performance",
        index=False
    )

    # 연도별 예측값 및 변수 중요도
    for year in rf_prediction_dict:

        rf_prediction_dict[year].to_excel(
            writer,
            sheet_name=f"Pred_{year}",
            index=False
        )

        rf_importance_dict[year].to_excel(
            writer,
            sheet_name=f"Importance_{year}",
            index=False
        )

    # 전체 기간 예측값
    rf_full_out["pred_df"].to_excel(
        writer,
        sheet_name="Pred_Full",
        index=False
    )

    # 전체 기간 변수 중요도
    rf_full_out["importance_df"].to_excel(
        writer,
        sheet_name="Importance_Full",
        index=False
    )

print(f"\n저장 완료: {output_path}")

Mounted at /content/drive
최종 설명변수 수: 22
['Size_m2', 'Floor', 'num_of_people', 'Parking_per_Household', 'Construction_Year', 'max_floor', 'Heating_Dummy', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD_km', 'Bus_Stop', 'High_School_Count', 'Population', 'Sex_ratio', 'Pop_Density', 'Median age Population', 'Old_pop_ratio', 'Young_pop_ratio', 'Spring', 'Fall', 'Winter']

[2022년 Random Forest 결과]
훈련 데이터셋
R²   : 0.9095
RMSE : 0.1353
테스트 데이터셋
R²   : 0.8371
RMSE : 0.1848
Train N : 7,405
Test N  : 3,174

변수 중요도 상위 10개


,Feature,Importance
0,num_of_people,0.148173
1,Young_pop_ratio,0.115190
2,Old_pop_ratio,0.107870
3,max_floor,0.083195
4,Construction_Year,0.073216
5,Population,0.060614
6,Sex_ratio,0.057339
7,Size_m2,0.042737
8,Heating_Dummy,0.041393
9,Parking_per_Household,0.039323



[2023년 Random Forest 결과]
훈련 데이터셋
R²   : 0.9456
RMSE : 0.1006
테스트 데이터셋
R²   : 0.9224
RMSE : 0.1208
Train N : 21,749
Test N  : 9,322

변수 중요도 상위 10개


,Feature,Importance
0,Young_pop_ratio,0.148565
1,Old_pop_ratio,0.126144
2,num_of_people,0.084122
3,Construction_Year,0.074538
4,Population,0.071417
5,max_floor,0.070613
6,Sex_ratio,0.044644
7,Pop_Density,0.042600
8,Heating_Dummy,0.042226
9,Median age Population,0.041007



[전체 기간 Random Forest 결과]
훈련 데이터셋
R²   : 0.9446
RMSE : 0.1090
테스트 데이터셋
R²   : 0.9321
RMSE : 0.1199
Train N : 113,662
Test N  : 48,713

전체 기간 변수 중요도 상위 10개


...


[2023년 예측값 일부]


,Actual,Predicted
0,16.454920,16.309726
1,16.102828,16.138045
2,16.659107,16.595432
3,16.075036,16.115074
4,16.046305,16.065169



[전체 기간 예측값 일부]


,Actual,Predicted
0,16.463642,16.535076
1,16.152554,16.148778
2,16.483932,16.002346
3,16.355151,16.336065
4,16.445963,16.454137



저장 완료: /content/drive/MyDrive/Seoul_Apartment_RF_Results_All_Features.xlsx
